In [ ]:
fire_dir = Path(
    "/Users/eem5633/Downloads/GeoTIFF_Qdeg_monthly_summaries/fire_number"
)

start_month = "200201"
end_month   = "200612"

monthly_files = []

for path in fire_dir.glob("Monthly_Qdeg_fire_number_*.tif"):
    match = re.search(r"(\d{6})\.tif$", path.name)

    if match is not None:
        year_month = match.group(1)

        if start_month <= year_month <= end_month:
            monthly_files.append((year_month, path))

monthly_files.sort(key=lambda item: item[0])

print("Number of files:", len(monthly_files))
print("First:", monthly_files[0])
print("Last:", monthly_files[-1])

assert len(monthly_files) == 60, (
    f"Expected 60 monthly files, but found {len(monthly_files)}"
)

Number of files: 60
First: ('200201', PosixPath('/Users/eem5633/Downloads/GeoTIFF_Qdeg_monthly_summaries/fire_number/Monthly_Qdeg_fire_number_200201.tif'))
Last: ('200612', PosixPath('/Users/eem5633/Downloads/GeoTIFF_Qdeg_monthly_summaries/fire_number/Monthly_Qdeg_fire_number_200612.tif'))


In [ ]:
monthly_grids = []
month_labels = []

reference_shape = None
reference_transform = None
reference_crs = None

for year_month, path in monthly_files:

    with rasterio.open(path) as src:
        grid = src.read(1).astype(np.float32)

        # Replace invalid and no-data values with zero
        grid[~np.isfinite(grid)] = 0.0

        if src.nodata is not None:
            grid[grid == src.nodata] = 0.0

        grid[grid < 0] = 0.0

        # Confirm that every file uses the same grid
        if reference_shape is None:
            reference_shape = src.shape
            reference_transform = src.transform
            reference_crs = src.crs
        else:
            assert src.shape == reference_shape
            assert src.transform == reference_transform
            assert src.crs == reference_crs

    monthly_grids.append(grid)

    # Convert "200201" into "2002-01"
    month_labels.append(
        f"{year_month[:4]}-{year_month[4:]}"
    )

monthly_grids = np.stack(monthly_grids)

print("Stack shape:", monthly_grids.shape)
print("Months:", month_labels[0], "through", month_labels[-1])

Stack shape: (60, 720, 1440)
Months: 2002-01 through 2006-12


In [ ]:
active_cells = np.any(monthly_grids > 0, axis=0)

rows, cols = np.nonzero(active_cells)

print("Cells active during at least one month:", len(rows))

Cells active during at least one month: 101457


In [ ]:
longitudes, latitudes = rasterio.transform.xy(
    reference_transform,
    rows,
    cols,
    offset="center"
)

longitudes = np.asarray(longitudes)
latitudes = np.asarray(latitudes)

In [ ]:
def latlon_to_xyz(longitude, latitude, radius):
    lon_rad = np.deg2rad(longitude)
    lat_rad = np.deg2rad(latitude)

    x = radius * np.cos(lat_rad) * np.cos(lon_rad)
    y = radius * np.cos(lat_rad) * np.sin(lon_rad)
    z = radius * np.sin(lat_rad)

    return np.column_stack((x, y, z))

In [ ]:
fire_radius = 101.0  # Use slightly more than your Earth radius

fire_coordinates = latlon_to_xyz(
    longitudes,
    latitudes,
    fire_radius
).astype(np.float32)

In [ ]:
monthly_fire_fields = []

for month_index in range(len(month_labels)):
    monthly_values = monthly_grids[month_index][active_cells]

    display_values = np.log10(1.0 + monthly_values)

    monthly_fire_fields.append(
        display_values.astype(np.float32)
    )

In [ ]:
number_of_fire_points = fire_coordinates.shape[0]

fire_colors = np.ones(

    (number_of_fire_points, 4),

    dtype=np.float32

)

fires_group = ParticleGroup(

    "Fires",

    fire_coordinates,

    rgba_colors=fire_colors,

    field_arrays=monthly_fire_fields,

    field_names=month_labels,

    field_filter_flags=[False] * len(month_labels),

    field_colormap_flags=[True] * len(month_labels),

)

Make sure each field_array (60) has a field_radius_flag (0), assuming False.


In [ ]:
reader.addParticleGroup(fires_group)

In [ ]:
all_displayed_values = np.concatenate(monthly_fire_fields)

color_min = 0.0
color_max = np.percentile(
    all_displayed_values[all_displayed_values > 0],
    99.5
)

print("Common color range:", color_min, color_max)

Common color range: 0.0 1.6822172


In [ ]:
common_limits = {
    label: [float(color_min), float(color_max)]
    for label in month_labels
}

reader.settings["colormapLims"] = {
    "Fires": common_limits
}

reader.settings["colormapVals"] = {
    "Fires": common_limits.copy()
}

reader.settings["showColormap"] = {
    "Earth": False,
    "Coast": False,
    "Fires": True,
}

reader.settings["colormapVariable"] = {
    "Earth": 0,
    "Coast": 0,
    "Fires": 0,       # Start with 2002-01
}

reader.settings["blendingMode"] = {
    "Earth": "normal",
    "Coast": "normal",
    "Fires": "additive",
}

In [ ]:
annual_2002_from_months = monthly_grids[:12].sum(axis=0)

print(
    "Estimated global total for 2002:",
    np.sum(annual_2002_from_months)
)

Estimated global total for 2002: 782373.25


======= socket connected


In [ ]:
# define the local port (typically anything in 5000 - 8000 range)
port = 6300

In [ ]:
# Note: you should only run this cell once.  You only want one single Firefly server running at time on your machine.
# If you've already run this cell and need to restart the server for some reason, the best thing to do is to restart your notebook kernel.
# (You can continue to send new data to the existing server, within needing to re-run this cell.)
process = spawnFireflyServer(port, method = 'flask',max_time=60)

Waiting up to 60 seconds for background Firefly server to start.....................

/opt/anaconda3/envs/firefly-dsfp/lib/python3.14/site-packages/firefly/server.py:24: EventletDeprecationWarning: 
Eventlet is deprecated. It is currently being maintained in bugfix mode, and
we strongly recommend against using it for new projects.

If you are already using Eventlet, we recommend migrating to a different
framework.  For more detail see
https://eventlet.readthedocs.io/en/latest/asyncio/migration.html

  from eventlet import event
 * Restarting with stat


Launching Firefly at: http://localhost:6300
from directory /opt/anaconda3/envs/firefly-dsfp/lib/python3.14/site-packages/firefly
..

/opt/anaconda3/envs/firefly-dsfp/lib/python3.14/site-packages/firefly/server.py:24: EventletDeprecationWarning: 
Eventlet is deprecated. It is currently being maintained in bugfix mode, and
we strongly recommend against using it for new projects.

If you are already using Eventlet, we recommend migrating to a different
framework.  For more detail see
https://eventlet.readthedocs.io/en/latest/asyncio/migration.html

  from eventlet import event


Launching Firefly at: http://localhost:6300
from directory /opt/anaconda3/envs/firefly-dsfp/lib/python3.14/site-packages/firefly
done! Your server is available at - http://localhost:6300


In [ ]:
from IPython.display import IFrame
url = f'http://localhost:{port:d}/combined'
IFrame(url, width=1000, height=500)

======= socket connected
======= connected {'data': 'GUI connected!'}
======= connected {'data': 'Viewer connected!'}
======= in room default_Firefly_AMG_ABG
======= in room default_Firefly_AMG_ABG


In [ ]:
# Send data to the server.
# Wait until it loads to run this command
reader.sendDataViaFlask(port=port)

Earth - 120000/120000 particles - 0 tracked fields
Coast - 5128/5128 particles - 0 tracked fields
Fires - 100928/100928 particles - 5 tracked fields
Fires - 101457/101457 particles - 60 tracked fields
Posting data on port 6300...======= receiving data from server ...
======= size of data 47918464
======= showing loader
======= sending data to viewer ...
/Users/eem5633/Documents/GitHub_repos/LSST-DSFP/WildFireFly-Hack_Session26/firefly_fires_2003_2007/DataEarth000.json 8388156
/Users/eem5633/Documents/GitHub_repos/LSST-DSFP/WildFireFly-Hack_Session26/firefly_fires_2003_2007/DataEarth001.json 1689129
/Users/eem5633/Documents/GitHub_repos/LSST-DSFP/WildFireFly-Hack_Session26/firefly_fires_2003_2007/DataCoast000.json 485071
/Users/eem5633/Documents/GitHub_repos/LSST-DSFP/WildFireFly-Hack_Session26/firefly_fires_2003_2007/DataFires000.json 36864064
/Users/eem5633/Documents/GitHub_repos/LSST-DSFP/WildFireFly-Hack_Session26/firefly_fires_2003_2007/DataFires001.json 484926
/Users/eem5633/Docum